# Tarea 1 · Parte 1 — Recolección y limpieza de noticias peruanas

**Pregunta guía:** ¿Pueden convertir titulares peruanos en un clasificador de noticias?

1. Configuración inicial: Registrar la fecha de descarga, el medio elegido (A, B o C) y las URLs base en la primera celda Markdown.

| Campo | Valor |
|---|---|
| Estudiante | Valeria Murga |
| Opción elegida | C – Comparada (La República + El Comercio) |
| Fecha de descarga | se registra en la celda de configuración |
| URLs base | https://larepublica.pe · https://elcomercio.pe |
| Fuente de datos | RSS y sitemaps oficiales (publicados en el robots.txt de cada diario) |

In [6]:
# ---------------- Importación ----------------
import os, re, time, html, hashlib
from datetime import datetime, date, timezone
from urllib import robotparser #Leer el robots.txt y comprobar que nos permiten descargar
from urllib.parse import urlparse
import xml.etree.ElementTree as ET  #Leer los RSS, que vienen en formato XML

import requests  #Descargar los feeds
import pandas as pd  #Armar y limpiar la tabla
from bs4 import BeautifulSoup #Limpiar el HTML que a veces viene dentro de los textos
import plotly.express as px #Los gráficos del EDA

print("Librerías listas ✓")

Librerías listas ✓


In [7]:
# ---------------- Configuración ----------------
OPCION = "C"    # "A" La República | "B" El Comercio | "C" Comparada
FECHA_EXTRACCION_UTC = datetime.now(timezone.utc).isoformat()  # momento exacto
FECHA_DESCARGA = date.today().isoformat() # solo la fecha (para nombres de archivo)
PAUSA_SEG = 1.5 # pausa entre solicitudes (1–2 s)
USER_AGENT = ("Mozilla/5.0 (compatible; TareaPUCP-QLAB/1.0; "
              "clasificacion de titulares; contacto: valeria.murga@pucp.edu.pe)")
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]

# Meta de noticias por medio y categoría (se recorta a las más recientes para balancear)
META_POR_CATEGORIA = {"A": 100, "B": 100, "C": 50}[OPCION]

DIR_DATOS = "datos"
DIR_RAW = os.path.join(DIR_DATOS, "raw")
os.makedirs(DIR_RAW, exist_ok=True)

print("Opción:", OPCION, "| Extracción (UTC):", FECHA_EXTRACCION_UTC, "| Meta por medio y categoría:", META_POR_CATEGORIA)

Opción: C | Extracción (UTC): 2026-10-17T22:56:12.615270+00:00 | Meta por medio y categoría: 50


2. Extracción: Extraer mínimo 350 noticias con las columnas: titular, bajada, medio, seccion_original, url, fecha. Usar RSS o HTML (requests + BeautifulSoup), implementando pausas entre peticiones y guardando respaldo crudo en datos/raw/.

    Fuentes: Cada fuente es un feed RSS o un sitemap de noticias de una sección. La sección del feed solo sirve para
saber dónde buscar; la `seccion_original` que se guarda se toma de la **ruta real de la URL** de cada nota
(por ejemplo `tecnologia/inteligencia-artificial`), y luego se mapea a las 4 categorías.

In [17]:
LR = "https://larepublica.pe"
EC = "https://elcomercio.pe"

FUENTES_LR = []
for sec in ["politica", "economia", "deportes", "ciencia"]:
    FUENTES_LR += [
        {"medio": "La Republica", "tipo": "rss",     "seccion_feed": sec, "url": f"{LR}/rss/{sec}.xml"},
        {"medio": "La Republica", "tipo": "sitemap", "seccion_feed": sec, "url": f"{LR}/sitemap/{sec}.xml"},
    ]

FUENTES_EC = []
for sec in ["politica", "economia", "deporte-total", "tecnologia"]:
    FUENTES_EC += [
        {"medio": "El Comercio", "tipo": "rss",     "seccion_feed": sec, "url": f"{EC}/arc/outboundfeeds/rss/category/{sec}/?outputType=xml"},
        {"medio": "El Comercio", "tipo": "rss",     "seccion_feed": sec, "url": f"{EC}/arc/outboundfeeds/google-news-feed/category/{sec}/?outputType=xml"},
        {"medio": "El Comercio", "tipo": "sitemap", "seccion_feed": sec, "url": f"{EC}/sitemap/news/{sec}/?outputType=xml"},
    ]

FUENTES = {"A": FUENTES_LR, "B": FUENTES_EC, "C": FUENTES_LR + FUENTES_EC}[OPCION]
pd.DataFrame(FUENTES)

,medio,tipo,seccion_feed,url
0,La Republica,rss,politica,https://larepublica.pe/rss/politica.xml
1,La Republica,sitemap,politica,https://larepublica.pe/sitemap/politica.xml
2,La Republica,rss,economia,https://larepublica.pe/rss/economia.xml
3,La Republica,sitemap,economia,https://larepublica.pe/sitemap/economia.xml
4,La Republica,rss,deportes,https://larepublica.pe/rss/deportes.xml
5,La Republica,sitemap,deportes,https://larepublica.pe/sitemap/deportes.xml
6,La Republica,rss,ciencia,https://larepublica.pe/rss/ciencia.xml
7,La Republica,sitemap,ciencia,https://larepublica.pe/sitemap/ciencia.xml
8,El Comercio,rss,politica,https://elcomercio.pe/arc/outboundfeeds/rss/ca...
9,El Comercio,rss,politica,https://elcomercio.pe/arc/outboundfeeds/google...


Descarga (robots.txt, User-Agent, pausas, 1 reintento, respaldo crudo)

In [18]:
sesion = requests.Session()
sesion.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "es-PE,es;q=0.9"})
_robots = {}
log_solicitudes = []

def permitido(url):
    """Consulta robots.txt del dominio (se cachea por dominio)."""
    base = f"{urlparse(url).scheme}://{urlparse(url).netloc}"
    if base not in _robots:
        rp = robotparser.RobotFileParser()
        try:
            r = sesion.get(base + "/robots.txt", timeout=20)
            rp.parse(r.text.splitlines())
        except Exception:
            rp = None
        _robots[base] = rp
        time.sleep(PAUSA_SEG)
    rp = _robots[base]
    return True if rp is None else rp.can_fetch(USER_AGENT, url)

def descargar(url, guardar_como=None):
    """GET con pausa y como máximo 1 reintento. Devuelve texto o None."""
    if not permitido(url):
        log_solicitudes.append({"url": url, "estado": "bloqueado_por_robots"})
        return None
    for intento in range(2):                       # intento inicial + 1 reintento
        try:
            r = sesion.get(url, timeout=25)
            time.sleep(PAUSA_SEG)
            log_solicitudes.append({"url": url, "estado": r.status_code, "intento": intento + 1})
            if r.status_code == 200:
                if guardar_como:
                    with open(os.path.join(DIR_RAW, guardar_como), "w", encoding="utf-8") as f:
                        f.write(r.text)
                return r.text
        except requests.RequestException as e:
            log_solicitudes.append({"url": url, "estado": f"error: {type(e).__name__}", "intento": intento + 1})
            time.sleep(PAUSA_SEG)
    return None

Procesamiento de RSS y sitemaps: Se utiliza xml.etree para extraer información de los archivos XML, sin considerar los espacios de nombres (media:, news:, dc:), lo que permite emplear el mismo código para ambos diarios.

In [19]:
def _local(tag):
    return tag.rsplit("}", 1)[-1]

def _hijo(elem, nombre):
    for c in elem:
        if _local(c.tag) == nombre:
            return c
    return None

def _txt(elem):
    return html.unescape((elem.text or "").strip()) if elem is not None else ""

def limpiar_texto(t):
    t = BeautifulSoup(t or "", "html.parser").get_text(" ")
    return re.sub(r"\s+", " ", html.unescape(t)).strip()

def parsear_rss(xml_texto):
    filas = []
    raiz = ET.fromstring(xml_texto.encode("utf-8"))
    for item in raiz.iter():
        if _local(item.tag) != "item":
            continue
        filas.append({
            "titular": limpiar_texto(_txt(_hijo(item, "title"))),
            "bajada":  limpiar_texto(_txt(_hijo(item, "description"))),
            "url":     _txt(_hijo(item, "link")),
            "fecha":   _txt(_hijo(item, "pubDate")),
        })
    return filas

def parsear_sitemap(xml_texto):
    filas = []
    raiz = ET.fromstring(xml_texto.encode("utf-8"))
    for u in raiz.iter():
        if _local(u.tag) != "url":
            continue
        loc = _txt(_hijo(u, "loc"))
        news = _hijo(u, "news")
        titulo = _txt(_hijo(news, "title")) if news is not None else ""
        fecha = _txt(_hijo(news, "publication_date")) if news is not None else _txt(_hijo(u, "lastmod"))
        if titulo:                                   # sitemaps sin título no sirven
            filas.append({"titular": limpiar_texto(titulo), "bajada": "", "url": loc, "fecha": fecha})
    return filas

In [20]:
registros = []
for f in FUENTES:
    nombre = f"{f['medio'].replace(' ', '')}_{f['tipo']}_{f['seccion_feed']}_{hashlib.md5(f['url'].encode()).hexdigest()[:6]}_{FECHA_DESCARGA}.xml"
    texto = descargar(f["url"], guardar_como=nombre)
    if texto is None:
        print("✗ sin respuesta:", f["url"]); continue
    try:
        filas = parsear_rss(texto) if f["tipo"] == "rss" else parsear_sitemap(texto)
    except ET.ParseError as e:
        print("✗ XML inválido:", f["url"], e); continue
    for x in filas:
        x.update({"medio": f["medio"], "fuente": f["tipo"], "seccion_feed": f["seccion_feed"], "fecha_extraccion_utc": FECHA_EXTRACCION_UTC})
    registros += filas
    print(f"✓ {len(filas):>4} entradas  {f['medio']:<13} {f['tipo']:<8} {f['url']}")

nuevo = pd.DataFrame(registros)
print("\nEntradas descargadas en esta corrida:", len(nuevo))

✓   20 entradas  La Republica  rss      https://larepublica.pe/rss/politica.xml
✓   20 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/politica.xml
✓    9 entradas  La Republica  rss      https://larepublica.pe/rss/economia.xml
✓    9 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/economia.xml
✓   12 entradas  La Republica  rss      https://larepublica.pe/rss/deportes.xml
✓   12 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/deportes.xml
✓   10 entradas  La Republica  rss      https://larepublica.pe/rss/ciencia.xml
✓   10 entradas  La Republica  sitemap  https://larepublica.pe/sitemap/ciencia.xml
✓   24 entradas  El Comercio   rss      https://elcomercio.pe/arc/outboundfeeds/rss/category/politica/?outputType=xml
✓   24 entradas  El Comercio   rss      https://elcomercio.pe/arc/outboundfeeds/google-news-feed/category/politica/?outputType=xml
✓  100 entradas  El Comercio   sitemap  https://elcomercio.pe/sitemap/news/politica/?outputType=x

Acumulación de noticias entre días: Los feeds RSS muestran únicamente las noticias más recientes. Por ello, en cada ejecución del notebook, las nuevas noticias se incorporan al archivo datos/raw/acumulado.csv. 

In [21]:
RUTA_ACUM = os.path.join(DIR_RAW, "acumulado.csv")
if os.path.exists(RUTA_ACUM):
    previo = pd.read_csv(RUTA_ACUM, encoding="utf-8-sig")
    acumulado = pd.concat([previo, nuevo], ignore_index=True)
else:
    acumulado = nuevo.copy()
acumulado = acumulado.drop_duplicates(subset=["url"], keep="first")
acumulado.to_csv(RUTA_ACUM, index=False, encoding="utf-8-sig")
print("Total acumulado (URLs únicas):", len(acumulado))
pd.DataFrame(log_solicitudes, columns=["url", "estado", "intento"])["estado"].value_counts()  # resumen de respuestas HTTP

Total acumulado (URLs únicas): 451


estado
200    20
Name: count, dtype: int64